# Homework 6: Mạng Nơ-ron Hồi quy (RNN) để Dự đoán Tên ở Cấp độ Ký tự
## Môn học: Trí Tuệ Nhân Tạo - EE3063

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Giới thiệu về RNN và bài toán dự đoán ký tự.
2. Chuẩn bị dữ liệu tên và mã hóa one-hot.
3. Xây dựng mô hình RNN đơn giản từ đầu.
4. Huấn luyện mô hình.
5. Dự đoán ký tự và sinh tên.

In [1]:
# -*- coding: utf-8 -*-
import numpy as np

## 1. Giới thiệu

**Mạng Nơ-ron Hồi quy (Recurrent Neural Network - RNN)** là một lớp các mạng nơ-ron nhân tạo được thiết kế đặc biệt để xử lý dữ liệu dạng chuỗi (sequential data), ví dụ như văn bản, chuỗi thời gian, âm thanh. Điểm đặc biệt của RNN là nó có các kết nối vòng (recurrent connections), cho phép thông tin từ các bước thời gian trước đó được "ghi nhớ" và ảnh hưởng đến việc xử lý ở các bước thời gian hiện tại và tương lai.

Trong bài tập này, chúng ta sẽ xây dựng một mô hình RNN đơn giản ở cấp độ ký tự (character-level) để học cách dự đoán ký tự tiếp theo trong một tên, dựa trên các ký tự đã xuất hiện trước đó. Ví dụ, nếu đầu vào là "B", mô hình sẽ cố gắng dự đoán "i"; nếu đầu vào là "Bi", mô hình sẽ cố gắng dự đoán "n", v.v.

Dataset cho bài toán này là một tập hợp nhỏ các tên: "Bình", "Long", "Dũng".

### Biểu diễn Dữ liệu
Mỗi ký tự sẽ được biểu diễn dưới dạng vector **one-hot encoding**. Ví dụ, nếu bộ từ vựng của chúng ta là {'B', 'ì', 'n', 'h', 'L', 'o', 'g', 'D', 'ũ'}, thì ký tự 'B' có thể được mã hóa thành [1,0,0,0,0,0,0,0,0], 'ì' thành [0,1,0,0,0,0,0,0,0], v.v.

### Kiến trúc RNN cơ bản
Mô hình RNN của chúng ta sẽ bao gồm:
1.  **Lớp đầu vào (Input Layer):** Nhận vector one-hot của ký tự hiện tại ($x_t$).
2.  **Lớp ẩn (Hidden Layer):** Tính toán trạng thái ẩn hiện tại ($h_t$) dựa trên $x_t$ và trạng thái ẩn trước đó ($h_{t-1}$).
    $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$
3.  **Lớp đầu ra (Output Layer):** Từ $h_t$, tính toán một vector điểm số (logits) cho mỗi ký tự trong bộ từ vựng.
    $o_t = W_{hy}h_t + b_y$
4.  **Hàm Softmax:** Chuyển đổi vector điểm số $o_t$ thành một phân phối xác suất $\hat{y}_t$ trên các ký tự, cho biết xác suất ký tự tiếp theo là gì.
    $\hat{y}_t = \text{softmax}(o_t)$

### Huấn luyện
-   **Hàm mất mát (Loss Function):** Sử dụng Cross-Entropy loss để so sánh phân phối xác suất dự đoán $\hat{y}_t$ với phân phối xác suất thực tế của ký tự tiếp theo (là một vector one-hot của ký tự đúng).
-   **Lan truyền ngược theo thời gian (Backpropagation Through Time - BPTT):** Tính toán gradient của hàm mất mát theo các tham số của mạng ($W_{xh}, W_{hh}, W_{hy}, b_h, b_y$) và cập nhật chúng bằng một thuật toán tối ưu (ví dụ: Stochastic Gradient Descent - SGD).

(Tham khảo slide: "RNN.pdf", "LSTM.pdf" - slide LSTM có giới thiệu chung về RNN và công thức cập nhật)

## 2. Chuẩn bị Dữ liệu

In [2]:
data = ["Bình", "Long", "Dũng"] # Thêm dấu kết thúc chuỗi (ví dụ: <EOS>) nếu cần thiết cho các bài toán phức tạp hơn
# Trong bài này, ta sẽ coi việc dự đoán ký tự cuối cùng là mục tiêu cuối.

# Tạo bộ từ vựng (vocabulary)
chars = set()
for name in data:
    for char in name:
        chars.add(char)

sorted_chars = sorted(list(chars))
char_to_int = {ch: i for i, ch in enumerate(sorted_chars)}
int_to_char = {i: ch for i, ch in enumerate(sorted_chars)}
vocab_size = len(sorted_chars)

print(f"Bộ từ vựng ({vocab_size} ký tự): {sorted_chars}")
print(f"Ánh xạ ký tự sang số: {char_to_int}")

# Tạo các cặp (input_sequence, target_char)
# Ví dụ: từ "Bình", ta có:
# "B" -> "ì"
# "Bì" -> "n"
# "Bìn" -> "h"
sequences = []
for name in data:
    for i in range(len(name)):
        # Input là one-hot của ký tự tại vị trí j
        # Target là one-hot của ký tự tại vị trí j+1
        # Mô hình sẽ học dự đoán ký tự tiếp theo dựa trên ký tự hiện tại và trạng thái ẩn
        # Hoặc, chính xác hơn cho RNN cơ bản: input là x_t, target là y_t (ký tự kế tiếp x_t)
        # và trạng thái ẩn h_{t-1} được truyền từ bước trước.
        # Chúng ta sẽ tạo các chuỗi đầu vào và đầu ra tương ứng.
        # Ví dụ: name = "Bình"
        # x_seq: [B, ì, n], y_seq: [ì, n, h]
        # Để đơn giản, ta sẽ làm theo kiểu:
        # (context_char_sequence, next_char)
        # B -> ì
        # B,ì -> n
        # B,ì,n -> h
        # Hoặc đơn giản hơn cho RNN vanilla:
        # Input: B, h_prev=0 -> Output: ì (target)
        # Input: ì, h_prev=h(B) -> Output: n (target)
        # Input: n, h_prev=h(ì) -> Output: h (target)

        # Chúng ta sẽ tạo dữ liệu theo từng ký tự một
        # Input: char_to_int[name[j]], Target: char_to_int[name[j+1]]
        for j in range(len(name) -1):
            input_char_idx = char_to_int[name[j]]
            target_char_idx = char_to_int[name[j+1]]
            sequences.append((input_char_idx, target_char_idx, name[:j+1])) # Lưu thêm context để dễ debug

print("\nCác cặp (input_idx, target_idx, context_string) được tạo:")
for seq in sequences:
    print(f"Input: {int_to_char[seq[0]]} ({seq[0]}), Target: {int_to_char[seq[1]]} ({seq[1]}), Context: '{seq[2]}'")

Bộ từ vựng (9 ký tự): ['B', 'D', 'L', 'g', 'h', 'n', 'o', 'ì', 'ũ']
Ánh xạ ký tự sang số: {'B': 0, 'D': 1, 'L': 2, 'g': 3, 'h': 4, 'n': 5, 'o': 6, 'ì': 7, 'ũ': 8}

Các cặp (input_idx, target_idx, context_string) được tạo:
Input: B (0), Target: ì (7), Context: 'B'
Input: ì (7), Target: n (5), Context: 'Bì'
Input: n (5), Target: h (4), Context: 'Bìn'
Input: B (0), Target: ì (7), Context: 'B'
Input: ì (7), Target: n (5), Context: 'Bì'
Input: n (5), Target: h (4), Context: 'Bìn'
Input: B (0), Target: ì (7), Context: 'B'
Input: ì (7), Target: n (5), Context: 'Bì'
Input: n (5), Target: h (4), Context: 'Bìn'
Input: B (0), Target: ì (7), Context: 'B'
Input: ì (7), Target: n (5), Context: 'Bì'
Input: n (5), Target: h (4), Context: 'Bìn'
Input: L (2), Target: o (6), Context: 'L'
Input: o (6), Target: n (5), Context: 'Lo'
Input: n (5), Target: g (3), Context: 'Lon'
Input: L (2), Target: o (6), Context: 'L'
Input: o (6), Target: n (5), Context: 'Lo'
Input: n (5), Target: g (3), Context: 'Lon'
Inpu

## 3. Xây dựng Mô hình RNN

In [5]:
# Hàm kích hoạt
def tanh(x):
    return np.tanh(x)

def softmax(x):
    # Trừ đi max(x) để tăng tính ổn định số học, tránh overflow khi exp(x) quá lớn
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True)) 
    return e_x / np.sum(e_x, axis=0, keepdims=True)

class SimpleRNN:
    def __init__(self, vocab_size, hidden_size, learning_rate=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate

        # Khởi tạo trọng số ngẫu nhiên (giá trị nhỏ)
        # W_xh: input to hidden
        self.W_xh = np.random.randn(hidden_size, vocab_size) * 0.01
        # W_hh: hidden to hidden (recurrent)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        # W_hy: hidden to output
        self.W_hy = np.random.randn(vocab_size, hidden_size) * 0.01
        
        # Biases
        self.b_h = np.zeros((hidden_size, 1)) # Bias cho lớp ẩn
        self.b_y = np.zeros((vocab_size, 1))  # Bias cho lớp output

    def forward(self, x_input_idx, h_prev):
        """
        Thực hiện một bước lan truyền xuôi.
        Args:
            x_input_idx (int): Chỉ số của ký tự đầu vào trong bộ từ vựng.
            h_prev (np.array): Trạng thái ẩn từ bước thời gian trước, shape (hidden_size, 1).
        Returns:
            y_pred_proba (np.array): Phân phối xác suất dự đoán cho ký tự tiếp theo.
            h_current (np.array): Trạng thái ẩn hiện tại.
            x_one_hot (np.array): Vector one-hot của input (để dùng trong backprop).
            o_logits (np.array): Điểm số (logits) trước softmax (để dùng trong backprop).
        """
        # Tạo vector one-hot cho ký tự đầu vào
        x_one_hot = np.zeros((self.vocab_size, 1))
        x_one_hot[x_input_idx] = 1
        
        # Tính trạng thái ẩn hiện tại
        # h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
        h_current = tanh(np.dot(self.W_xh, x_one_hot) + np.dot(self.W_hh, h_prev) + self.b_h)
        
        # Tính điểm số (logits) cho lớp output
        # o_t = W_hy * h_t + b_y
        o_logits = np.dot(self.W_hy, h_current) + self.b_y
        
        # Tính xác suất đầu ra bằng softmax
        y_pred_proba = softmax(o_logits)
        
        return y_pred_proba, h_current, x_one_hot, o_logits

    def train_sequence(self, input_char_indices, target_char_indices):
        """
        Huấn luyện RNN trên một chuỗi các ký tự.
        Args:
            input_char_indices (list of int): Danh sách chỉ số của các ký tự đầu vào.
            target_char_indices (list of int): Danh sách chỉ số của các ký tự mục tiêu.
        Returns:
            float: Giá trị loss trung bình trên chuỗi.
            dict: Gradients của các tham số.
        """
        # Khởi tạo
        loss = 0
        h_prev = np.zeros((self.hidden_size, 1)) # Trạng thái ẩn ban đầu
        
        # Lưu trữ các giá trị trung gian cho backpropagation
        xs, hs, os, ys_pred = {}, {}, {}, {} 
        hs[-1] = np.copy(h_prev) # Lưu trạng thái ẩn ban đầu

        # --- Lan truyền xuôi (Forward pass) ---
        for t in range(len(input_char_indices)):
            x_idx = input_char_indices[t]
            y_target_idx = target_char_indices[t]
            
            y_pred_proba_t, h_current_t, x_one_hot_t, o_logits_t = self.forward(x_idx, hs[t-1])
            
            # Lưu trữ
            xs[t] = x_one_hot_t
            hs[t] = h_current_t
            os[t] = o_logits_t
            ys_pred[t] = y_pred_proba_t
            
            # Tính Cross-Entropy loss cho bước thời gian t
            # Loss = - sum(y_true * log(y_pred))
            # y_true ở đây là one-hot của target_char_idx
            loss_t = -np.log(y_pred_proba_t[y_target_idx, 0] + 1e-9) # Thêm 1e-9 để tránh log(0)
            loss += loss_t
        
        # --- Lan truyền ngược theo thời gian (BPTT) ---
        # Khởi tạo gradients
        dW_xh, dW_hh, dW_hy = np.zeros_like(self.W_xh), np.zeros_like(self.W_hh), np.zeros_like(self.W_hy)
        db_h, db_y = np.zeros_like(self.b_h), np.zeros_like(self.b_y)
        
        dh_next = np.zeros_like(hs[0]) # Gradient của loss theo h_t, truyền từ bước t+1 về t

        for t in reversed(range(len(input_char_indices))):
            y_target_idx = target_char_indices[t]
            
            # Gradient của loss theo output logits o_t (dL/do_t)
            # dL/do_t = y_pred_t - y_true_t (đạo hàm của Softmax + CrossEntropy)
            dy = np.copy(ys_pred[t])
            dy[y_target_idx] -= 1 
            
            # Gradient cho W_hy và b_y
            # dL/dW_hy = dL/do_t * do_t/dW_hy = dy * h_t^T
            dW_hy += np.dot(dy, hs[t].T)
            # dL/db_y = dL/do_t * do_t/db_y = dy
            db_y += dy
            
            # Gradient của loss theo hidden state h_t (dL/dh_t)
            # dL/dh_t = dL/do_t * do_t/dh_t + dL/dh_{t+1} * dh_{t+1}/dh_t (từ BPTT)
            #            = W_hy^T * dy      + dh_next
            dh = np.dot(self.W_hy.T, dy) + dh_next
            
            # Gradient của hàm tanh: dtanh = (1 - h_t^2)
            # dL/d(raw_h_input) = dL/dh_t * dh_t/d(raw_h_input) = dh * (1 - hs[t]^2)
            dh_raw = (1 - hs[t]**2) * dh # Phần tử theo phần tử
            
            # Gradient cho b_h
            db_h += dh_raw
            
            # Gradient cho W_xh
            # dL/dW_xh = dL/d(raw_h_input) * d(raw_h_input)/dW_xh = dh_raw * x_t^T
            dW_xh += np.dot(dh_raw, xs[t].T)
            
            # Gradient cho W_hh
            # dL/dW_hh = dL/d(raw_h_input) * d(raw_h_input)/dW_hh = dh_raw * h_{t-1}^T
            dW_hh += np.dot(dh_raw, hs[t-1].T)
            
            # Cập nhật dh_next cho bước lặp ngược tiếp theo
            # dL/dh_{t-1} = dL/d(raw_h_input) * d(raw_h_input)/dh_{t-1} = W_hh^T * dh_raw
            dh_next = np.dot(self.W_hh.T, dh_raw)

        # Giới hạn gradient (gradient clipping) để tránh exploding gradients (tùy chọn)
        for dparam in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(dparam, -5, 5, out=dparam) 
            
        avg_loss = loss / len(input_char_indices)
        gradients = {'dW_xh': dW_xh, 'dW_hh': dW_hh, 'dW_hy': dW_hy, 'db_h': db_h, 'db_y': db_y}
        
        return avg_loss, gradients

    def update_parameters(self, gradients):
        self.W_xh -= self.lr * gradients['dW_xh']
        self.W_hh -= self.lr * gradients['dW_hh']
        self.W_hy -= self.lr * gradients['dW_hy']
        self.b_h  -= self.lr * gradients['db_h']
        self.b_y  -= self.lr * gradients['db_y']

## 4. Huấn luyện Mô hình RNN

Chúng ta sẽ huấn luyện mô hình trên các chuỗi ký tự từ bộ dữ liệu tên.

In [6]:
hidden_size_rnn = 50 # Số lượng units trong lớp ẩn, có thể điều chỉnh
learning_rate_rnn = 0.01
n_epochs_rnn = 500 # Số epochs huấn luyện

rnn_model = SimpleRNN(vocab_size, hidden_size_rnn, learning_rate_rnn)

print(f"\nBắt đầu huấn luyện RNN với hidden_size={hidden_size_rnn}, lr={learning_rate_rnn}...")

for epoch in range(n_epochs_rnn):
    total_loss_epoch = 0
    
    # Huấn luyện trên từng tên (chuỗi)
    for name_idx, name_str in enumerate(data):
        input_indices = [char_to_int[ch] for ch in name_str[:-1]] # Tất cả trừ ký tự cuối
        target_indices = [char_to_int[ch] for ch in name_str[1:]] # Tất cả trừ ký tự đầu
        
        if not input_indices: # Nếu tên chỉ có 1 ký tự, không có cặp input-target
            continue
            
        loss, grads = rnn_model.train_sequence(input_indices, target_indices)
        rnn_model.update_parameters(grads)
        total_loss_epoch += loss
        
    avg_loss_epoch = total_loss_epoch / len(data)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{n_epochs_rnn}, Loss trung bình: {avg_loss_epoch:.4f}")

print("Hoàn tất huấn luyện RNN.")


Bắt đầu huấn luyện RNN với hidden_size=50, lr=0.01...
Epoch 100/500, Loss trung bình: 1.7765
Epoch 200/500, Loss trung bình: 1.0366
Epoch 300/500, Loss trung bình: 0.5048
Epoch 400/500, Loss trung bình: 0.2850
Epoch 500/500, Loss trung bình: 0.0679
Hoàn tất huấn luyện RNN.


## 5. Dự đoán Ký tự và Sinh Tên

Sau khi huấn luyện, chúng ta có thể sử dụng mô hình để:
1.  Dự đoán ký tự tiếp theo cho một chuỗi ký tự đầu vào.
2.  Sinh ra một chuỗi tên mới bằng cách lấy mẫu từ phân phối xác suất đầu ra.

In [10]:
def predict_next_char(model, start_string, h_prev_init=None):
    """Dự đoán ký tự tiếp theo cho một chuỗi bắt đầu."""
    if h_prev_init is None:
        h = np.zeros((model.hidden_size, 1))
    else:
        h = h_prev_init
        
    # Đưa chuỗi start_string qua mô hình để cập nhật trạng thái ẩn h
    for char_str in start_string:
        if char_str not in char_to_int:
            print(f"Ký tự '{char_str}' không có trong bộ từ vựng.")
            return None, h # Trả về trạng thái ẩn cuối cùng
        x_idx = char_to_int[char_str]
        _, h, _, _ = model.forward(x_idx, h) # Chỉ cần h cuối cùng
        
    # Ký tự cuối cùng của start_string sẽ là input cho dự đoán ký tự tiếp theo
    # (Hoặc có thể hiểu là h đã chứa thông tin của start_string,
    # và ký tự tiếp theo được dự đoán từ h này và một input "start token" nếu có,
    # hoặc đơn giản là dự đoán từ h cuối cùng của start_string)

    # Để dự đoán ký tự *sau* start_string, ta cần input là ký tự cuối của start_string
    # và trạng thái ẩn h *trước* khi xử lý ký tự cuối đó.
    # Hoặc, nếu mô hình học "ký tự tiếp theo của cả chuỗi", thì h hiện tại là đúng.
    
    # Giả sử ta muốn dự đoán ký tự sau ký tự cuối cùng của start_string
    # Ta cần trạng thái ẩn h sau khi xử lý toàn bộ start_string
    # và input là ký tự cuối cùng của start_string để dự đoán ký tự kế tiếp nó.
    # Tuy nhiên, hàm forward hiện tại dự đoán ký tự kế tiếp của x_input_idx.
    # Nên nếu start_string là "Bi", h là trạng thái sau "i",
    # và y_pred_proba là dự đoán cho ký tự sau "i".

    if not start_string:
        print("Chuỗi bắt đầu rỗng.")
        # Có thể dự đoán ký tự đầu tiên nếu có một "start token" hoặc h_prev_init khác 0
        # Ở đây, ta trả về None nếu không có start_string
        return None, h

    last_char_idx = char_to_int[start_string[-1]] # Lấy ký tự cuối làm input cho bước forward cuối cùng
                                                # để lấy y_pred_proba cho ký tự *sau* nó.
                                                # Tuy nhiên, h đã được cập nhật bởi ký tự cuối rồi.
                                                # Cách tiếp cận đúng hơn là:
                                                # h_after_start_string là hs[len(start_string)-1]
                                                # y_pred_proba là ys_pred[len(start_string)-1]
                                                # Điều này cần thay đổi cách hàm train_sequence trả về và lưu trữ.

    # Đơn giản hóa: dùng h hiện tại (sau khi xử lý toàn bộ start_string)
    # và dự đoán dựa trên một input "dummy" hoặc ký tự cuối.
    # Trong cách triển khai hiện tại của forward, nó sẽ cho ra xác suất của ký tự
    # *sau* ký tự cuối cùng của start_string nếu input cho forward là ký tự cuối đó.
    
    # Lấy lại ký tự cuối của start_string làm input cho bước dự đoán cuối cùng
    # để y_pred_proba là dự đoán cho ký tự SAU ký tự cuối đó.
    final_input_idx = char_to_int[start_string[-1]]
    y_pred_proba, _, _, _ = model.forward(final_input_idx, h) # h ở đây là trạng thái sau khi xử lý start_string[-1]

    # Lấy ký tự có xác suất cao nhất
    predicted_idx = np.argmax(y_pred_proba.flatten())
    return int_to_char[predicted_idx], h # Trả về cả trạng thái ẩn cuối để có thể sinh tiếp


def complete_name(model, start_char_str, max_length=5):
    """Hoàn thành tên bắt đầu bằng start_char_str."""
    if start_char_str not in char_to_int:
        print(f"Ký tự bắt đầu '{start_char_str}' không có trong bộ từ vựng.")
        return start_char_str
        
    current_name = start_char_str
    h = np.zeros((model.hidden_size, 1)) # Trạng thái ẩn ban đầu

    # "Mồi" trạng thái ẩn bằng ký tự bắt đầu
    # (Hoặc có thể coi ký tự bắt đầu là input đầu tiên cho vòng lặp sinh)
    # Cách 1: Xử lý ký tự bắt đầu để có h_init cho vòng lặp sinh
    # _, h, _, _ = model.forward(char_to_int[start_char_str], h)
    # current_char_idx = char_to_int[start_char_str] # Ký tự input cho vòng lặp sẽ là ký tự vừa được dự đoán

    # Cách 2: Bắt đầu sinh từ ký tự đầu tiên
    current_char_idx = char_to_int[start_char_str]

    for _ in range(max_length - len(start_char_str)):
        y_pred_proba, h_next, _, _ = model.forward(current_char_idx, h)
        
        # Chọn ký tự có xác suất cao nhất
        next_char_idx = np.argmax(y_pred_proba.flatten())
        # Hoặc có thể lấy mẫu (np.random.choice) từ y_pred_proba để có sự đa dạng hơn
        # next_char_idx = np.random.choice(range(model.vocab_size), p=y_pred_proba.flatten())

        next_char = int_to_char[next_char_idx]
        current_name += next_char
        
        # Cập nhật cho bước tiếp theo
        current_char_idx = next_char_idx
        h = h_next
        
        # Có thể dừng nếu gặp một ký tự đặc biệt (ví dụ: <EOS>) nếu có
        if len(current_name) >= len(data[0]) + 2 : # Giới hạn độ dài sinh, tránh quá dài
            break
            
    return current_name

print("\n--- Dự đoán và Hoàn thành Tên ---")
# Dự đoán cho các tên trong dataset
target_names = ["Bình", "Long", "Dũng"]
start_chars = ["B", "L", "D"]

for i, start_char in enumerate(start_chars):
    completed = complete_name(rnn_model, start_char, max_length=len(target_names[i]))
    print(f"Bắt đầu bằng '{start_char}', Hoàn thành: '{completed}' (Mong muốn: '{target_names[i]}')")

# Thử với các chuỗi con khác
test_contexts = ["Bi", "Lo", "Dũ"]
for context in test_contexts:
    h_context = np.zeros((rnn_model.hidden_size, 1))
    # "Prime" the hidden state with the context
    for char_ctx in context[:-1]: # Tất cả trừ ký tự cuối
         _, h_context, _, _ = rnn_model.forward(char_to_int[char_ctx], h_context)
    
    # Dự đoán ký tự tiếp theo cho ký tự cuối của context
    next_char_pred, _ = predict_next_char(rnn_model, context[-1], h_prev_init=h_context)
    if next_char_pred:
        print(f"Sau chuỗi '{context}', ký tự dự đoán tiếp theo: '{next_char_pred}'")


--- Dự đoán và Hoàn thành Tên ---
Bắt đầu bằng 'B', Hoàn thành: 'Bình' (Mong muốn: 'Bình')
Bắt đầu bằng 'L', Hoàn thành: 'Long' (Mong muốn: 'Long')
Bắt đầu bằng 'D', Hoàn thành: 'Dũng' (Mong muốn: 'Dũng')
Ký tự 'i' không có trong bộ từ vựng.
Sau chuỗi 'Lo', ký tự dự đoán tiếp theo: 'g'
Sau chuỗi 'Dũ', ký tự dự đoán tiếp theo: 'g'


## 6. Nhận xét và Kết luận

-   Mô hình RNN đơn giản ở cấp độ ký tự đã được xây dựng từ đầu, bao gồm các bước lan truyền xuôi, tính toán hàm mất mát (cross-entropy), và lan truyền ngược theo thời gian (BPTT) để cập nhật trọng số.
-   Mô hình được huấn luyện trên một tập dữ liệu nhỏ gồm ba tên ("Bình", "Long", "Dũng").
-   **Kết quả dự đoán:**
    * Với tập dữ liệu rất nhỏ này và mô hình RNN đơn giản, khả năng dự đoán chính xác hoàn toàn các tên có thể bị hạn chế. Mô hình có thể học được một số quy luật cơ bản về các cặp ký tự thường xuất hiện (ví dụ: sau 'B' thường là 'ì').
    * Giá trị `hidden_size`, `learning_rate`, và số `epochs` ảnh hưởng lớn đến kết quả. Cần tinh chỉnh các tham số này để có kết quả tốt hơn.
    * Việc khởi tạo trọng số ngẫu nhiên cũng có thể dẫn đến các kết quả khác nhau giữa các lần chạy.
-   **Hạn chế của mô hình RNN cơ bản:**
    * **Vanishing/Exploding Gradients:** Với các chuỗi dài, RNN cơ bản gặp khó khăn trong việc học các phụ thuộc xa do vấn đề gradient biến mất hoặc bùng nổ. Bài toán này với các tên ngắn có thể ít bị ảnh hưởng nghiêm trọng.
    * **Khả năng nhớ ngắn hạn:** Bộ nhớ của RNN cơ bản có giới hạn.
-   **Hướng cải thiện (cho các bài toán phức tạp hơn):**
    * Sử dụng các kiến trúc RNN phức tạp hơn như LSTM hoặc GRU (sẽ được đề cập trong HW7).
    * Tăng kích thước tập dữ liệu huấn luyện.
    * Tinh chỉnh các siêu tham số (hyperparameters).
    * Sử dụng các kỹ thuật regularization (ví dụ: dropout) nếu mô hình bị overfitting (khó xảy ra với tập dữ liệu nhỏ này).